# 17 — Optimisation et choix du modele final

In [ ]:
from pathlib import Path
import sys
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

warnings.filterwarnings("ignore")
sys.path.append(r"..\configs")
from config_projet import C

dossiers = sorted(C.COHORTE.glob("cible_ancree_359_*"))
assert dossiers, "Dossier cible_ancree_359_* introuvable : lance d'abord le notebook 07c."
SRC = dossiers[-1]
print("Source :", SRC.name)

jeu = pd.read_csv(SRC / "jeu_modelisation_359_ancre.csv")
print("Lignes du fichier :", len(jeu))

# Population d'interet : patients PAS deja dements, dont l'avenir est connu.
arisque = jeu.loc[(jeu["deja_dement"] == 0) & jeu["progression_ancree"].notna()].copy()
y = arisque["progression_ancree"].astype(int).to_numpy()
print(f"Population a risque : {len(arisque)} | progressent : {y.sum()} ({y.mean():.1%})")

## 1 — Variables explicatives

In [ ]:
# Colonnes a exclure pour cause de FUITE DE DONNEES (voir le texte au-dessus).
FUITES = {"progression_ancienne", "suivi_apres_T0_jours"}

EXCLURE_EXACT = {
    "PTID", "RID", "SITEID", "VISCODE", "VISCODE2", "DIAGNOSIS", "DXCHANGE",
    "progression", "progression_ancree", "cible", "deja_dement",
    "dx_T0", "dx_futur_max", "n_visites_futures", "annee_naissance",
} | FUITES
EXCLURE_PREFIXE = ("date", "T0", "derniere", "chemin", "fichier", "naissance", "Unnamed")

def exclue(col):
    return col in EXCLURE_EXACT or col.startswith(EXCLURE_PREFIXE)

num = [c for c in arisque.columns
       if not exclue(c) and pd.api.types.is_numeric_dtype(arisque[c])]
cat = [c for c in arisque.columns
       if not exclue(c) and not pd.api.types.is_numeric_dtype(arisque[c])
       and arisque[c].nunique(dropna=True) <= 5]

Xdf = arisque[num].copy()
for c in cat:
    muettes = pd.get_dummies(arisque[c].astype("string"), prefix=c, drop_first=True)
    Xdf = pd.concat([Xdf, muettes.astype(float)], axis=1)

variables = list(Xdf.columns)
X = Xdf.to_numpy(dtype=float)
print("Colonnes ecartees pour fuite :", sorted(FUITES & set(jeu.columns)))
print(f"\n{len(variables)} variables explicatives "
      f"({len(num)} numeriques + {len(cat)} categorielles encodees)")
print(variables)
print("\nTaux de valeurs manquantes (5 plus eleves) :")
print((Xdf.isna().mean() * 100).sort_values(ascending=False).head(5).round(1).to_string())

# --- garde-fou anti-fuite ---
corr = Xdf.apply(lambda s: s.corr(pd.Series(y, index=s.index))).abs().sort_values(ascending=False)
print("\nCorrelation absolue avec la cible (5 plus fortes) :")
print(corr.head(5).round(3).to_string())
if corr.max() > 0.95:
    print("!! ALERTE : correlation > 0.95 — une colonne-cible est restee dans les variables.")
elif corr.max() > 0.80:
    print("!  Vigilance : correlation > 0.80, verifie le sens de cette variable.")
else:
    print("-> OK, aucune variable suspecte.")

## 1bis — Diagnostic : combien rapportait la fuite ?On entraine volontairement un modele **sur les colonnes fuitees seules**, avec le meme protocole.Le but n'est pas de garder ce modele mais de **chiffrer** le probleme : c'est la preuve que lafuite a ete detectee et mesuree, pas simplement devinee.Lecture attendue : une AUC proche de 1 pour `progression_ancienne` (elle contient la reponse),et une AUC nettement superieure au hasard pour `suivi_apres_T0_jours` (biais d'horizon de suivi).

In [ ]:
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

_CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)

def _auc_avec(colonnes):
    Xf = arisque[colonnes].to_numpy(dtype=float)
    tuyau = Pipeline([("imp", SimpleImputer(strategy="median")),
                      ("ech", StandardScaler()),
                      ("clf", LogisticRegression(max_iter=5000, class_weight="balanced"))])
    s = cross_val_score(tuyau, Xf, y, cv=_CV, scoring="roc_auc")
    return s.mean(), s.std()

diagnostic = []
for col in sorted(FUITES & set(arisque.columns)):
    m, e = _auc_avec([col])
    diagnostic.append({"variable fuitee (seule)": col,
                       "ROC-AUC": round(m, 3), "ecart-type": round(e, 3)})
dispo = sorted(FUITES & set(arisque.columns))
if len(dispo) > 1:
    m, e = _auc_avec(dispo)
    diagnostic.append({"variable fuitee (seule)": " + ".join(dispo),
                       "ROC-AUC": round(m, 3), "ecart-type": round(e, 3)})

diagnostic = pd.DataFrame(diagnostic)
print("=== Ce qu'aurait donne un modele nourri UNIQUEMENT de colonnes fuitees ===")
display(diagnostic)
print("Rappel : hasard = 0.500. Ces modeles sont REJETES, ils servent de diagnostic.")

## 2 — Validation croisee imbriquee + modele d'ensemble

In [ ]:
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

EXTERNE = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)   # identique aux autres notebooks
INTERNE = StratifiedKFold(n_splits=4, shuffle=True, random_state=1)   # pour la recherche de grille

def tuyau_logreg():
    return Pipeline([("imp", SimpleImputer(strategy="median")),
                     ("ech", StandardScaler()),
                     ("clf", LogisticRegression(max_iter=5000, class_weight="balanced"))])

def tuyau_foret():
    return Pipeline([("imp", SimpleImputer(strategy="median")),
                     ("clf", RandomForestClassifier(n_estimators=300, random_state=0,
                                                    class_weight="balanced", n_jobs=-1))])

GRILLE_LOGREG = {"clf__C": [0.003, 0.01, 0.03, 0.1, 0.3, 1, 3, 10, 30]}
GRILLE_FORET = {"clf__max_depth": [None, 4, 8],
                "clf__min_samples_leaf": [1, 3, 8],
                "clf__max_features": ["sqrt", 0.5]}

NOMS = ["Regression logistique (par defaut)", "Random forest (par defaut)",
        "Regression logistique (reglee)", "Random forest (reglee)",
        "Ensemble (moyenne des deux reglees)"]

auc_par_pli = {n: [] for n in NOMS}
oof = {n: np.full(len(y), np.nan) for n in NOMS}   # probabilites hors echantillon
params = {NOMS[2]: [], NOMS[3]: []}

for k, (app, test) in enumerate(EXTERNE.split(X, y), 1):
    p = {}
    p[NOMS[0]] = tuyau_logreg().fit(X[app], y[app]).predict_proba(X[test])[:, 1]
    p[NOMS[1]] = tuyau_foret().fit(X[app], y[app]).predict_proba(X[test])[:, 1]

    g1 = GridSearchCV(tuyau_logreg(), GRILLE_LOGREG, scoring="roc_auc",
                      cv=INTERNE, n_jobs=-1).fit(X[app], y[app])
    p[NOMS[2]] = g1.predict_proba(X[test])[:, 1]
    params[NOMS[2]].append(g1.best_params_)

    g2 = GridSearchCV(tuyau_foret(), GRILLE_FORET, scoring="roc_auc",
                      cv=INTERNE, n_jobs=-1).fit(X[app], y[app])
    p[NOMS[3]] = g2.predict_proba(X[test])[:, 1]
    params[NOMS[3]].append(g2.best_params_)

    p[NOMS[4]] = (p[NOMS[2]] + p[NOMS[3]]) / 2

    for n in NOMS:
        auc_par_pli[n].append(roc_auc_score(y[test], p[n]))
        oof[n][test] = p[n]
    print(f"  pli {k}/5 termine")

assert not any(np.isnan(v).any() for v in oof.values()), "probabilites hors echantillon incompletes"
print("\nTermine.")

## 3 — Classement et effet du reglage

In [ ]:
SORTIE = C.sortie("modele_final")

classement = (pd.DataFrame({
    "modele": NOMS,
    "roc_auc_moyenne": [np.mean(auc_par_pli[n]) for n in NOMS],
    "roc_auc_ecart_type": [np.std(auc_par_pli[n]) for n in NOMS],
}).sort_values("roc_auc_moyenne", ascending=False).reset_index(drop=True))

print("=== Classement (ROC-AUC, 5 plis externes) ===")
display(classement.round(3))

par_pli = pd.DataFrame(auc_par_pli, index=[f"pli {i}" for i in range(1, 6)])
print("\n=== Detail par pli ===")
display(par_pli.round(3))

# Effet du reglage, compare pli par pli (le plus honnete).
print("\n=== Effet du reglage des hyperparametres ===")
for base, regle in [(NOMS[0], NOMS[2]), (NOMS[1], NOMS[3])]:
    ecart = par_pli[regle] - par_pli[base]
    print(f"{base.split(' (')[0]:22s} : {par_pli[base].mean():.3f} -> {par_pli[regle].mean():.3f}"
          f"  ({ecart.mean():+.3f} ± {ecart.std():.3f}, gagne {int((ecart > 0).sum())}/5 plis)")

ecart_ens = par_pli[NOMS[4]] - par_pli[[NOMS[2], NOMS[3]]].max(axis=1)
print(f"{'Ensemble vs meilleur seul':22s} : {ecart_ens.mean():+.3f} ± {ecart_ens.std():.3f}"
      f"  (gagne {int((ecart_ens > 0).sum())}/5 plis)")

print("\n=== Hyperparametres choisis par pli ===")
for n in [NOMS[2], NOMS[3]]:
    print(f"\n{n} :")
    for i, d in enumerate(params[n], 1):
        print(f"  pli {i} : " + ", ".join(f"{k.replace('clf__', '')}={v}" for k, v in d.items()))

classement.to_csv(SORTIE / "classement_modele_final.csv", index=False, encoding="utf-8-sig")
par_pli.round(4).to_csv(SORTIE / "AUC_par_pli_modele_final.csv", encoding="utf-8-sig")

## 4 — Seuil de decision et performance operationnelleL'AUC resume la qualite du **classement** des patients, mais pour decider il faut un **seuil**.Avec environ un tiers de progresseurs, le seuil 0,50 n'est pas forcement le bon choix.- **Youden** : maximise `sensibilite + specificite - 1` → equilibre.- **sensibilite ≥ 80 %** : logique de depistage, on accepte plus de fausses alertes  pour ne pas manquer de patients qui vont s'aggraver.C'est aussi ici qu'apparait l'**exactitude** (accuracy) : a comparer au score d'un modelequi repondrait toujours « ne progresse pas ».

In [ ]:
from sklearn.metrics import roc_curve, confusion_matrix, balanced_accuracy_score

MEILLEUR = classement.loc[0, "modele"]
proba = oof[MEILLEUR]
print("Modele evalue :", MEILLEUR)
print(f"Reference naive (toujours « ne progresse pas ») : exactitude = {1 - y.mean():.3f}, "
      f"exactitude equilibree = 0.500\n")

fpr, tpr, seuils = roc_curve(y, proba)
i_youden = int(np.argmax(tpr - fpr))
seuil_youden = float(seuils[i_youden])
ok = np.where(tpr >= 0.80)[0]
seuil_sens80 = float(seuils[ok[0]]) if len(ok) else seuil_youden

def mesures(s):
    pred = (proba >= s).astype(int)
    vn, fp, fn, vp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    return {"seuil": round(s, 3),
            "sensibilite": round(vp / max(vp + fn, 1), 3),
            "specificite": round(vn / max(vn + fp, 1), 3),
            "VPP": round(vp / max(vp + fp, 1), 3),
            "VPN": round(vn / max(vn + fn, 1), 3),
            "exactitude equilibree": round(balanced_accuracy_score(y, pred), 3),
            "exactitude": round((vp + vn) / len(y), 3)}

seuils_table = pd.DataFrame([
    {"strategie": "0,50 (par defaut)", **mesures(0.50)},
    {"strategie": "Youden (equilibre)", **mesures(seuil_youden)},
    {"strategie": "sensibilite >= 80 %", **mesures(seuil_sens80)},
])
display(seuils_table)
seuils_table.to_csv(SORTIE / "seuils_et_mesures.csv", index=False, encoding="utf-8-sig")

print("\n=== Matrice de confusion au seuil de Youden "
      f"({seuil_youden:.3f}) ===")
pred = (proba >= seuil_youden).astype(int)
mat = pd.DataFrame(confusion_matrix(y, pred, labels=[0, 1]),
                   index=["reel : ne progresse pas", "reel : progresse"],
                   columns=["predit : ne progresse pas", "predit : progresse"])
display(mat)
mat.to_csv(SORTIE / "matrice_confusion.csv", encoding="utf-8-sig")

## 5 — Figure de synthese

In [ ]:
from sklearn.calibration import calibration_curve

fig, (a1, a2, a3) = plt.subplots(1, 3, figsize=(15, 4.2), constrained_layout=True)

ordre = classement.sort_values("roc_auc_moyenne")
pos = np.arange(len(ordre))
a1.barh(pos, ordre["roc_auc_moyenne"], xerr=ordre["roc_auc_ecart_type"], capsize=3,
        color=["#4C72B0" if m == MEILLEUR else "#B0B0B0" for m in ordre["modele"]])
a1.set_yticks(pos)
a1.set_yticklabels([m.replace(" (", "\n(") for m in ordre["modele"]], fontsize=7)
a1.axvline(0.5, color="grey", ls="--", lw=1)
a1.set_xlim(0.45, 1.0)
for p, m, e in zip(pos, ordre["roc_auc_moyenne"], ordre["roc_auc_ecart_type"]):
    a1.text(m + e + 0.012, p, f"{m:.3f}", va="center", fontsize=8)
a1.set_xlabel("ROC-AUC (5 plis)")
a1.set_title("Comparaison des modeles", fontsize=10)

a2.plot(fpr, tpr, color="#4C72B0", lw=2,
        label=f"AUC = {classement.loc[0, 'roc_auc_moyenne']:.3f}")
a2.plot([0, 1], [0, 1], color="grey", ls="--", lw=1, label="hasard")
a2.scatter([fpr[i_youden]], [tpr[i_youden]], color="#C44E52", s=70, zorder=5)
a2.annotate(f"seuil {seuil_youden:.2f}", (fpr[i_youden], tpr[i_youden]),
            textcoords="offset points", xytext=(10, -14), fontsize=8, color="#C44E52")
a2.set_xlabel("1 - specificite")
a2.set_ylabel("sensibilite")
a2.set_title("Courbe ROC du modele retenu", fontsize=10)
a2.legend(fontsize=8, loc="lower right")

frac, moyenne = calibration_curve(y, proba, n_bins=5, strategy="quantile")
a3.plot(moyenne, frac, "o-", color="#4C72B0")
a3.plot([0, 1], [0, 1], color="grey", ls="--", lw=1)
a3.set_xlabel("probabilite predite")
a3.set_ylabel("frequence observee")
a3.set_title("Calibration (5 groupes)", fontsize=10)

fig.savefig(SORTIE / "modele_final_figure.png", dpi=150)
plt.show()
print("Figure :", SORTIE / "modele_final_figure.png")

## 6 — Conclusion

In [ ]:
diagnostic.to_csv(SORTIE / "diagnostic_fuite.csv", index=False, encoding="utf-8-sig")

best = classement.loc[0]
ref = classement.loc[classement["modele"] == "Regression logistique (par defaut)"].iloc[0]
gain = best["roc_auc_moyenne"] - ref["roc_auc_moyenne"]

# --- Controle de coherence avec le notebook 11b ---
d11b = sorted(C.COHORTE.glob("comparaison_modeles_ancree_*"))
if d11b:
    t = pd.read_csv(d11b[-1] / "classement_AUC_ancree.csv")
    t = t.loc[t["modele"] != "Hasard (reference)"]
    meilleur_11b = t.sort_values("roc_auc_moyenne", ascending=False).iloc[0]
    ecart = ref["roc_auc_moyenne"] - meilleur_11b["roc_auc_moyenne"]
    print(f"Controle de coherence avec le notebook 11b :")
    print(f"  11b, meilleur modele : {meilleur_11b['modele']} = "
          f"{meilleur_11b['roc_auc_moyenne']:.3f}")
    print(f"  17, meme modele par defaut : {ref['roc_auc_moyenne']:.3f}  (ecart {ecart:+.3f})")
    if abs(ecart) < 0.03:
        print("  -> coherent : les deux notebooks mesurent bien la meme chose.\n")
    else:
        print("  -> ECART IMPORTANT : verifie que 11b exclut aussi les colonnes fuitees.\n")

print("=" * 78)
print(f">>> MODELE FINAL RETENU : {best['modele']}")
print(f"    ROC-AUC = {best['roc_auc_moyenne']:.3f} ± {best['roc_auc_ecart_type']:.3f} "
      f"(N = {len(y)}, {y.mean():.1%} de progresseurs)")
print(f"    Gain apporte par le reglage / l'ensemble : {gain:+.3f}")
print(f"    Seuil conseille (Youden) = {seuil_youden:.3f} -> sensibilite "
      f"{mesures(seuil_youden)['sensibilite']:.2f}, specificite "
      f"{mesures(seuil_youden)['specificite']:.2f}, "
      f"exactitude equilibree {mesures(seuil_youden)['exactitude equilibree']:.2f}")
print("=" * 78)
if abs(gain) < 0.02:
    print("\nLecture : le reglage n'apporte pas de gain notable. C'est un resultat en soi —")
    print("la performance est limitee par la taille de l'echantillon et par la difficulte")
    print("intrinseque de la tache, pas par le choix des hyperparametres.")

resume = pd.DataFrame([{
    "modele_final": best["modele"],
    "roc_auc": round(best["roc_auc_moyenne"], 3),
    "ecart_type": round(best["roc_auc_ecart_type"], 3),
    "N": len(y),
    "taux_progresseurs": round(float(y.mean()), 3),
    "seuil_conseille": round(seuil_youden, 3),
    **{k: v for k, v in mesures(seuil_youden).items() if k != "seuil"},
}])
resume.to_csv(SORTIE / "modele_final_resume.csv", index=False, encoding="utf-8-sig")
display(resume)
print("\nTout est enregistre dans :", SORTIE)